# Corpus Quality Analysis (in-process)

Interactively run `Orchestrator.analyze_hierarchical` against the
labelled COVID-19 corpus in `fixtures/analyze_corpus.yaml`, using
the **live LLM** but **no FastAPI server**.

This complements `scripts/stress_analyze.py` (which exercises the
real REST path for load testing). Here the goal is to study a single
analysis end-to-end: inspect the synthesis, judge score, confidence,
and the deterministic `coding_trends` table.

### Prerequisites

- `LLM_API_KEY` set in the environment (or in `.env`) the same way the
  server reads it.
- The BGE-M3 embedding model, required for hierarchical mode. Fetch it once
  with `uv run python scripts/fetch_embedding_model.py` and put the three
  `EMBEDDING_*` lines it prints into `.env` (relative `.models/...` paths
  work — the next cell anchors the working directory to the repo root). If
  you skip this, switch `mode` below to `single_pass` and use a small
  `limit` (the corpus is well past the single-pass token cap, so single_pass
  will 413 above ~tens of records).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%%time
import os
from pathlib import Path

# Anchor the working directory to the repo root *before* anything reads a
# relative path. The notebook lives in notebooks/, but the API server runs
# from the repo root, so relative paths in .env (e.g.
# EMBEDDING_MODEL_PATH=.models/bge-m3-onnx-int8/model_quantized.onnx) and the
# fixtures path below only resolve correctly when CWD is the repo root. Walk
# up to the first ancestor containing pyproject.toml.
_repo_root = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()
)
os.chdir(_repo_root)

import dotenv

dotenv.load_dotenv(dotenv.find_dotenv(usecwd=True), override=True)

## 1. Imports and setup

In [ ]:
import sys
from datetime import UTC, datetime, timedelta
from pathlib import Path

import polars as pl
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

# CWD was anchored to the repo root in the first cell, so cwd() is the root.
# Make ``scripts/`` importable to share the loader/sampler with stress_analyze.py.
REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "scripts"))

from stress_analyze import load_sample  # noqa: E402

from qfa.api.composition import build_orchestrator  # noqa: E402
from qfa.domain.models import AnalysisRequestModel, FeedbackRecordModel  # noqa: E402
from qfa.settings import AppSettings  # noqa: E402
from qfa.utils import timed  # noqa: E402

In [ ]:
# Activate qfa's logging so the per-phase timings emitted by
# ``analyze_hierarchical`` (anonymisation, embedding, clustering, map, reduce)
# and the per-call LLM latency (DEBUG) show up inline beneath the run cell.
#
# ``force=True`` is required in Jupyter: ipykernel installs a root log handler
# at kernel startup, and ``logging.basicConfig`` is a no-op once a handler
# already exists — ``force`` tears down that handler and installs ours, so the
# project's format and levels actually take effect. ``LogSettings`` defaults
# qfa to DEBUG (per-call + per-chunk detail) and third-party libraries to
# WARNING; set ``LOG_LOGLEVEL=info`` in ``.env`` if the per-call lines are too
# chatty and you only want the phase summaries.
from qfa.settings import LogSettings  # noqa: E402
from qfa.utils import setup_logging  # noqa: E402

setup_logging(LogSettings(basicConfig={**LogSettings().basicConfig, "force": True}))

## 2. Compose the orchestrator

`build_orchestrator` constructs the same domain object graph the
FastAPI lifespan builds, minus the `TrackingLLMAdapter` (no DB needed
for ad-hoc analysis). The first call also registers custom LiteLLM
model prices, so `result.cost` reflects real billing for any
self-hosted model in `qfa/resources/model_prices.yaml`.

In [ ]:
%%time
settings = AppSettings()
orchestrator = build_orchestrator(settings)

# Fail fast if hierarchical mode can't run. The orchestrator would otherwise
# raise AnalysisError deep in cell 4 — better to surface the env requirement
# now, before any LLM call has been made.
if orchestrator._embedder is None:
    raise RuntimeError(
        "EMBEDDING_MODEL_PATH is unset — hierarchical analysis requires a local "
        "BGE-M3 ONNX export. Set EMBEDDING_MODEL_PATH (and EMBEDDING_REVISION_HASH) "
        "in your shell or .env, the same way the API server reads them, then "
        "restart the kernel. See docs/architecture/07-hierarchical-analysis.md "
        "for the model setup."
    )

print(
    f"LLM:        {settings.llm.model}\n"
    f"Embedder:   configured\n"
    f"Max tokens: {settings.llm.max_total_tokens:,}"
)

## 3. Sample a small batch

We're after a small-enough sample that the call returns in a couple
of minutes but large-enough that the hierarchical pipeline actually
chunks (i.e. exceeds the single-pass cap). 50 records is a good
starting point; bump it up to study how chunking and reduction scale.

In [ ]:
%%time
LIMIT = 5000
SEED = 42

raw_records = load_sample(REPO_ROOT / "fixtures" / "analyze_corpus.yaml", LIMIT, SEED)
feedback_records = tuple(
    FeedbackRecordModel(id=r["id"], text=r["text"], metadata=r["metadata"])
    for r in raw_records
)
print(f"Sampled {len(feedback_records)} records (seed={SEED}).")
print(f"First record id: {feedback_records[0].id}")

## 4. Run `analyze_hierarchical`

The deadline is generous (5 minutes) because hierarchical mode
fires one LLM call per cluster plus a reduce call. With anonymization
enabled, Presidio is invoked before every LLM call.

_Note: this cell makes real LLM calls and costs real money._

In [ ]:
request = AnalysisRequestModel(
    feedback_records=feedback_records,
    prompt=(
        "Identify the main themes, sentiments, and any trending concerns "
        "in this feedback corpus. Highlight signals that change over time."
    ),
    tenant_id="notebook-quality-run",
    mode="hierarchical",
)
deadline = datetime.now(UTC) + timedelta(minutes=5)

# No ``%%time`` here: that cell magic recompiles its body without CPython's
# PyCF_ALLOW_TOP_LEVEL_AWAIT flag, so a top-level ``await`` raises
# ``SyntaxError: 'await' outside function``. ``timed()`` is a plain (sync)
# context manager — entering and exiting it is synchronous, the ``await`` just
# lives inside the block — so it sidesteps that limitation and reuses the same
# stopwatch the orchestrator uses internally. The per-phase breakdown
# (anonymisation, embedding, clustering, map, reduce) prints in the log output
# above; this is just the end-to-end wall time.
with timed() as run:
    result = await orchestrator.analyze_hierarchical(request, deadline)
print(f"analyze_hierarchical wall time: {run.elapsed_seconds:.1f}s")

## 5. Inspect the synthesis

`result.result` is the analyst-facing text (already de-anonymised);
`quality_score` and `uncertainty_explanation` come from the AI-as-judge
pass; `confidence` is the coverage-weighted mean of per-chunk judge
scores during the map phase.

In [ ]:
print(f"quality_score (judge):   {result.quality_score}")
print(f"confidence (map mean):   {result.confidence}")
print(f"uncertainty:             {result.uncertainty_explanation}\n")
display(Markdown(result.result))

## 6. Plot the coding-trend table

`coding_trends` is built deterministically from record metadata —
it does **not** depend on the LLM. We pivot it into a code × period
matrix and plot the codes with the strongest temporal variation.
The planted patterns in this corpus (spike, step, emerging,
declining) should be visible when the sample is large enough; with
only 50 records the signal is noisy.

In [ ]:
trends = result.coding_trends
if trends is None or not trends.cells:
    print("No coding_trends produced for this sample.")
else:
    df = pl.DataFrame(
        [{"code": c.code, "period": c.period, "count": c.count} for c in trends.cells]
    )
    # Pick the 8 codes with the strongest temporal variation. ``std`` over
    # the per-period counts per code: spike/step/emerging/declining patterns
    # have high std; flat baseline codes have low std and stay out of the plot.
    top_codes = (
        df.group_by("code")
        .agg(pl.col("count").std().alias("std"))
        .sort("std", descending=True, nulls_last=True)
        .head(8)
        .get_column("code")
        .to_list()
    )
    # Wide format: one row per code, one column per period.
    plot_df = (
        df.filter(pl.col("code").is_in(top_codes))
        .pivot(values="count", index="code", on="period", aggregate_function="sum")
        .fill_null(0)
    )
    period_cols = [p for p in trends.periods if p in plot_df.columns]
    codes = plot_df.get_column("code").to_list()
    values = plot_df.select(period_cols).to_numpy()

    fig, ax = plt.subplots(figsize=(10, 5))
    for i, code in enumerate(codes):
        ax.plot(period_cols, values[i], marker="o", label=code)
    ax.set_xticks(range(len(period_cols)))
    ax.set_xticklabels(period_cols, rotation=45, ha="right")
    ax.set_ylabel("records")
    ax.set_title(f"coding_trends — top 8 codes by std (n={len(feedback_records)})")
    ax.legend(bbox_to_anchor=(1.04, 1), loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.show()

## 7. (Optional) Tweak settings and re-run

Because the orchestrator is a plain Python object here, you can
construct a second one with different `analyze_settings` (e.g.
`min_cluster_size`) to see how chunking changes. Useful when
the synthesis above looks too granular or too coarse.

In [ ]:
# Example: smaller clusters → more chunks → more map calls (higher cost,
# potentially more nuance). Uncomment to try.
#
# from qfa.settings import AnalyzeSettings
#
# tweaked_settings = AppSettings()
# tweaked_settings.analyze = AnalyzeSettings(min_cluster_size=3)
# tweaked_orchestrator = build_orchestrator(tweaked_settings)
# deadline = datetime.now(UTC) + timedelta(minutes=5)
# alt = await tweaked_orchestrator.analyze_hierarchical(request, deadline)
# display(Markdown(alt.result))